# Native pytket circuits and optimization levels

Build circuits directly with `pytket.Circuit`, then apply different
optimization passes at levels 0 through 4. Shows how gate count and
depth decrease with each pass.

In [ ]:
from pytket import Circuit
from pytket.passes import (
    FullPeepholeOptimise,
    RemoveRedundancies,
    CommuteThroughMultis,
    CliffordSimp,
)

In [ ]:
def circuit_info(c: Circuit) -> dict:
    ops = {}
    for g in c.get_commands():
        name = g.op.type.__name__
        ops[name] = ops.get(name, 0) + 1
    return {"gates": c.n_gates, "depth": c.depth(), "ops": dict(sorted(ops.items()))}


def optimize(c: Circuit, level: int) -> Circuit:
    c_opt = c.copy()
    if level >= 1:
        RemoveRedundancies().apply(c_opt)
    if level >= 2:
        CommuteThroughMultis().apply(c_opt)
        RemoveRedundancies().apply(c_opt)
    if level >= 3:
        CliffordSimp().apply(c_opt)
        RemoveRedundancies().apply(c_opt)
    if level >= 4:
        FullPeepholeOptimise().apply(c_opt)
        RemoveRedundancies().apply(c_opt)
    return c_opt


def demo(name: str, c: Circuit) -> None:
    print(f"\n{'=' * 50}")
    print(f"  {name}")
    print(f"{'=' * 50}")
    info = circuit_info(c)
    print(f"  Level 0 (original): gates={info['gates']}  depth={info['depth']}")
    print(f"    ops={info['ops']}")
    for level in range(1, 5):
        c_opt = optimize(c, level)
        info = circuit_info(c_opt)
        print(f"  Level {level}: gates={info['gates']}  depth={info['depth']}")
        print(f"    ops={info['ops']}")

## Bell circuit (2 qubits)

Minimal circuit — not much to optimize, but shows the pipeline.

In [ ]:
bell = Circuit(2)
bell.H(0)
bell.CX(0, 1)
demo("Bell circuit", bell)

## Entangled with redundancies (3 qubits)

Same pattern as the cross-framework examples: redundant pairs that
should collapse.

In [ ]:
c = Circuit(3)
c.H(0)
c.CX(0, 1)
c.CX(1, 2)
c.X(0)
c.X(0)
c.H(1)
c.H(1)
c.CX(2, 0)
c.T(0)
c.Tdg(0)
demo("Entangled + redundant", c)

## Variational ansatz (3 qubits)

A typical VQE-style circuit with rotation gates and entangling layers.
Rotation gates are harder to eliminate, but the optimizer can still
reduce depth.

In [ ]:
v = Circuit(3)
v.Rx(0.5, 0)
v.Ry(0.3, 1)
v.Rz(0.7, 2)
v.CX(0, 1)
v.CX(1, 2)
v.Rx(0.2, 0)
v.Ry(0.4, 1)
v.Rz(0.6, 2)
v.CX(0, 2)
v.H(0)
v.S(1)
v.T(2)
demo("Variational ansatz", v)